# 02 — Embeddings and Vector Store

## Responsible AI RAG Assistant

This notebook creates the embedding and vector-search layer for the Responsible AI RAG Assistant project.

The previous notebook prepared raw source text and processed chunks for the first real source:

`SRC-001 — European Commission AI Act overview page`

This notebook will load those processed chunks, generate vector embeddings, store them in a local Chroma vector database, and test semantic retrieval.

## Notebook Goals

By the end of this notebook, we want to:

1. Load processed chunk files.
2. Validate chunk structure and metadata.
3. Generate text embeddings using a local sentence-transformer model.
4. Store embeddings in a local Chroma vector database.
5. Test semantic search queries.
6. Prepare the retrieval layer for the RAG assistant.

## Important Note

This notebook uses local embeddings first.

No private documents, API keys, `.env` files, raw source files, or generated vector-store files should be uploaded to GitHub.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import date

print("Notebook 02 setup started.")
print(f"Current date: {date.today()}")

Notebook 02 setup started.
Current date: 2026-06-30


In [2]:
# Project paths

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
VECTOR_STORE_DIR = DATA_DIR / "vector_store"
REPORTS_DIR = PROJECT_ROOT / "reports"

paths = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "DATA_DIR": DATA_DIR,
    "PROCESSED_DATA_DIR": PROCESSED_DATA_DIR,
    "VECTOR_STORE_DIR": VECTOR_STORE_DIR,
    "REPORTS_DIR": REPORTS_DIR,
}

for name, path in paths.items():
    print(f"{name}: {path}")
    print(f"Exists: {path.exists()}")
    print("-" * 80)

PROJECT_ROOT: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant
Exists: True
--------------------------------------------------------------------------------
DATA_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data
Exists: True
--------------------------------------------------------------------------------
PROCESSED_DATA_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed
Exists: True
--------------------------------------------------------------------------------
VECTOR_STORE_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\vector_store
Exists: True
--------------------------------------------------------------------------------
REPORTS_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\reports
Exists: True
--------------------------------------------------------------------------------


## Load Processed Chunks

The first notebook saved processed chunks for the European Commission AI Act overview page.

In this section, we load the processed chunk table and validate that it contains the metadata required for retrieval and citation display.

In [3]:
# Load processed chunks from Notebook 01

src_001_chunks_path = (
    PROCESSED_DATA_DIR / "src_001_european_commission_ai_act_overview_chunks.csv"
)

if not src_001_chunks_path.exists():
    raise FileNotFoundError(f"Processed chunk file not found: {src_001_chunks_path}")

chunks_df = pd.read_csv(src_001_chunks_path)

print(f"Loaded chunk file: {src_001_chunks_path}")
print(f"Number of chunks: {len(chunks_df)}")
print(f"Columns: {list(chunks_df.columns)}")

chunks_df.head()

Loaded chunk file: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\src_001_european_commission_ai_act_overview_chunks.csv
Number of chunks: 11
Columns: ['source_id', 'source_name', 'publisher', 'source_type', 'url', 'planned_raw_filename', 'chunk_id', 'chunk_index', 'chunk_text', 'word_count']


,source_id,source_name,publisher,source_type,url,planned_raw_filename,chunk_id,chunk_index,chunk_text,word_count
0,SRC-001,AI Act,European Commission,Official web page,https://digital-strategy.ec.europa.eu/en/polic...,src_001_european_commission_ai_act_overview.txt,SRC-001_CHUNK_001,1,The AI Act is the first-ever legal framework o...,250
1,SRC-001,AI Act,European Commission,Official web page,https://digital-strategy.ec.europa.eu/en/polic...,src_001_european_commission_ai_act_overview.txt,SRC-001_CHUNK_002,2,"of the AI Act ahead of time. In parallel, the ...",250
2,SRC-001,AI Act,European Commission,Official web page,https://digital-strategy.ec.europa.eu/en/polic...,src_001_european_commission_ai_act_overview.txt,SRC-001_CHUNK_003,3,"prohibits eight practices , namely: harmful AI...",250
3,SRC-001,AI Act,European Commission,Official web page,https://digital-strategy.ec.europa.eu/en/polic...,src_001_european_commission_ai_act_overview.txt,SRC-001_CHUNK_004,4,professional life (e.g. scoring of exams) AI-b...,250
4,SRC-001,AI Act,European Commission,Official web page,https://digital-strategy.ec.europa.eu/en/polic...,src_001_european_commission_ai_act_overview.txt,SRC-001_CHUNK_005,5,"high level of robustness, cybersecurity and ac...",250


In [4]:
# Validate processed chunk table

required_columns = [
    "source_id",
    "source_name",
    "publisher",
    "source_type",
    "url",
    "planned_raw_filename",
    "chunk_id",
    "chunk_index",
    "chunk_text",
    "word_count",
]

missing_columns = [col for col in required_columns if col not in chunks_df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if chunks_df.empty:
    raise ValueError("chunks_df is empty.")

if not chunks_df["chunk_id"].is_unique:
    raise ValueError("chunk_id values are not unique.")

if chunks_df["chunk_text"].isna().any():
    raise ValueError("Some chunks have missing text.")

if (chunks_df["word_count"] <= 0).any():
    raise ValueError("Some chunks have zero or negative word counts.")

print("Processed chunk validation passed.")
print(f"Validated chunks: {len(chunks_df)}")
print(f"Total words across chunks: {chunks_df['word_count'].sum()}")
print(f"Average words per chunk: {chunks_df['word_count'].mean():.1f}")
print(f"Sources included: {chunks_df['source_id'].nunique()}")

Processed chunk validation passed.
Validated chunks: 11
Total words across chunks: 2729
Average words per chunk: 248.1
Sources included: 1


## Embedding Model

Embeddings convert text chunks into numerical vectors.

These vectors allow the system to search for semantically similar chunks, even when the exact words in the user question do not match the document text.

For this portfolio version, the project first uses a local sentence-transformer model:

`sentence-transformers/all-MiniLM-L6-v2`

This model is small, practical, and suitable for demonstrating the retrieval layer of a RAG system.

In [5]:
from sentence_transformers import SentenceTransformer

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading embedding model: {embedding_model_name}")

embedding_model = SentenceTransformer(embedding_model_name)

print("Embedding model loaded successfully.")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mdadg\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [6]:
# Test embedding generation on one chunk

sample_chunk_text = chunks_df.loc[0, "chunk_text"]

sample_embedding = embedding_model.encode(sample_chunk_text)

print(f"Sample chunk length in words: {len(sample_chunk_text.split())}")
print(f"Embedding type: {type(sample_embedding)}")
print(f"Embedding shape: {sample_embedding.shape}")
print(f"First 10 embedding values:")
print(sample_embedding[:10])

Sample chunk length in words: 250
Embedding type: <class 'numpy.ndarray'>
Embedding shape: (384,)
First 10 embedding values:
[-3.8367860e-02 -6.3510746e-02  2.7370146e-03 -7.3454119e-02
  6.9103822e-02  6.9743745e-02  1.0670103e-02  4.6510965e-02
 -4.7601650e-05  5.6069471e-02]


## Generate Embeddings for All Chunks

Now that the embedding model works on one sample chunk, the next step is to generate embeddings for all processed chunks.

Each chunk will be converted into a 384-dimensional vector using the local sentence-transformer model.

These embeddings will later be stored in a Chroma vector database for semantic search.

In [7]:
# Generate embeddings for all chunks

chunk_texts = chunks_df["chunk_text"].astype(str).tolist()

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"Number of chunk texts: {len(chunk_texts)}")
print(f"Embedding matrix type: {type(chunk_embeddings)}")
print(f"Embedding matrix shape: {chunk_embeddings.shape}")
print(f"Embedding dimension: {chunk_embeddings.shape[1]}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of chunk texts: 11
Embedding matrix type: <class 'numpy.ndarray'>
Embedding matrix shape: (11, 384)
Embedding dimension: 384


In [8]:
# Validate embedding output

if len(chunk_embeddings) != len(chunks_df):
    raise ValueError("Number of embeddings does not match number of chunks.")

if chunk_embeddings.shape[1] != 384:
    raise ValueError(f"Unexpected embedding dimension: {chunk_embeddings.shape[1]}")

if np.isnan(chunk_embeddings).any():
    raise ValueError("Embedding matrix contains NaN values.")

print("Embedding validation passed.")
print(f"Validated embeddings: {chunk_embeddings.shape[0]}")
print(f"Embedding dimension: {chunk_embeddings.shape[1]}")
print(f"Minimum value: {chunk_embeddings.min():.4f}")
print(f"Maximum value: {chunk_embeddings.max():.4f}")

Embedding validation passed.
Validated embeddings: 11
Embedding dimension: 384
Minimum value: -0.1723
Maximum value: 0.2017


## Create Chroma Vector Store

The processed chunks and their embeddings will now be stored in a local Chroma vector database.

Chroma allows the project to retrieve semantically relevant chunks for a user question.

The vector store is saved locally in:

`data/vector_store/`

This folder is ignored by Git and should not be uploaded to GitHub.

In [9]:
import chromadb

# Create local persistent Chroma client

chroma_client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))

collection_name = "responsible_ai_rag_src_001"

# Recreate collection for reproducibility during development
try:
    chroma_client.delete_collection(name=collection_name)
    print(f"Deleted existing collection: {collection_name}")
except Exception:
    print(f"No existing collection found with name: {collection_name}")

collection = chroma_client.create_collection(
    name=collection_name,
    metadata={
        "description": "Responsible AI RAG chunks from SRC-001 European Commission AI Act overview page"
    }
)

print(f"Created collection: {collection_name}")

No existing collection found with name: responsible_ai_rag_src_001
Created collection: responsible_ai_rag_src_001


In [10]:
# Prepare IDs, documents, metadata, and embeddings for Chroma

chroma_ids = chunks_df["chunk_id"].astype(str).tolist()
chroma_documents = chunks_df["chunk_text"].astype(str).tolist()

chroma_metadatas = []

for _, row in chunks_df.iterrows():
    chroma_metadatas.append(
        {
            "source_id": str(row["source_id"]),
            "source_name": str(row["source_name"]),
            "publisher": str(row["publisher"]),
            "source_type": str(row["source_type"]),
            "url": str(row["url"]),
            "planned_raw_filename": str(row["planned_raw_filename"]),
            "chunk_index": int(row["chunk_index"]),
            "word_count": int(row["word_count"]),
        }
    )

chroma_embeddings = chunk_embeddings.astype(float).tolist()

print(f"Prepared IDs: {len(chroma_ids)}")
print(f"Prepared documents: {len(chroma_documents)}")
print(f"Prepared metadata records: {len(chroma_metadatas)}")
print(f"Prepared embeddings: {len(chroma_embeddings)}")
print()
print("Example ID:", chroma_ids[0])
print("Example metadata:", chroma_metadatas[0])

Prepared IDs: 11
Prepared documents: 11
Prepared metadata records: 11
Prepared embeddings: 11

Example ID: SRC-001_CHUNK_001
Example metadata: {'source_id': 'SRC-001', 'source_name': 'AI Act', 'publisher': 'European Commission', 'source_type': 'Official web page', 'url': 'https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai', 'planned_raw_filename': 'src_001_european_commission_ai_act_overview.txt', 'chunk_index': 1, 'word_count': 250}


In [11]:
# Add chunks to Chroma vector store

collection.add(
    ids=chroma_ids,
    documents=chroma_documents,
    metadatas=chroma_metadatas,
    embeddings=chroma_embeddings,
)

print("Chunks added to Chroma collection.")
print(f"Collection count: {collection.count()}")

Chunks added to Chroma collection.
Collection count: 11


## Test Semantic Retrieval

Now the vector database can be tested with AI governance questions.

The goal is to check whether the system retrieves chunks that are semantically relevant to the question, even when the wording is not exactly the same as the source text.

For this first test, the vector store contains only one source:

`SRC-001 — European Commission AI Act overview page`

Later, the retrieval system will become stronger after adding GDPR, EUR-Lex AI Act text, and NIST AI RMF sources.

In [12]:
def search_vector_store(query: str, n_results: int = 3) -> pd.DataFrame:
    """
    Search the Chroma vector store using a natural language query.
    
    Parameters
    ----------
    query : str
        User question or search query.
    n_results : int
        Number of retrieved chunks.
    
    Returns
    -------
    pd.DataFrame
        Retrieved chunks with metadata and distance scores.
    """
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).astype(float).tolist()
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )
    
    records = []
    
    for rank, (doc, metadata, distance) in enumerate(
        zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        ),
        start=1,
    ):
        records.append(
            {
                "rank": rank,
                "distance": distance,
                "source_id": metadata["source_id"],
                "source_name": metadata["source_name"],
                "chunk_index": metadata["chunk_index"],
                "word_count": metadata["word_count"],
                "url": metadata["url"],
                "retrieved_text": doc,
            }
        )
    
    return pd.DataFrame(records)


print("Vector search helper function is ready.")

Vector search helper function is ready.


In [13]:
query_1 = "What is the risk-based approach of the EU AI Act?"

results_query_1 = search_vector_store(query_1, n_results=3)

print(f"Query: {query_1}")

results_query_1[
    [
        "rank",
        "distance",
        "source_id",
        "source_name",
        "chunk_index",
        "word_count",
        "retrieved_text",
    ]
]

Query: What is the risk-based approach of the EU AI Act?


,rank,distance,source_id,source_name,chunk_index,word_count,retrieved_text
0,1,0.404169,SRC-001,AI Act,2,250,"of the AI Act ahead of time. In parallel, the ..."
1,2,0.537072,SRC-001,AI Act,1,250,The AI Act is the first-ever legal framework o...
2,3,0.711476,SRC-001,AI Act,10,250,"on general-purpose AI models, reducing governa..."


In [14]:
query_2 = "What obligations apply to high-risk AI systems?"

results_query_2 = search_vector_store(query_2, n_results=3)

print(f"Query: {query_2}")

results_query_2[
    [
        "rank",
        "distance",
        "source_id",
        "source_name",
        "chunk_index",
        "word_count",
        "retrieved_text",
    ]
]

Query: What obligations apply to high-risk AI systems?


,rank,distance,source_id,source_name,chunk_index,word_count,retrieved_text
0,1,0.588744,SRC-001,AI Act,2,250,"of the AI Act ahead of time. In parallel, the ..."
1,2,0.782707,SRC-001,AI Act,4,250,professional life (e.g. scoring of exams) AI-b...
2,3,0.789766,SRC-001,AI Act,5,250,"high level of robustness, cybersecurity and ac..."


In [15]:
query_3 = "Does the AI Act require transparency for AI-generated content?"

results_query_3 = search_vector_store(query_3, n_results=3)

print(f"Query: {query_3}")

results_query_3[
    [
        "rank",
        "distance",
        "source_id",
        "source_name",
        "chunk_index",
        "word_count",
        "retrieved_text",
    ]
]

Query: Does the AI Act require transparency for AI-generated content?


,rank,distance,source_id,source_name,chunk_index,word_count,retrieved_text
0,1,0.613211,SRC-001,AI Act,5,250,"high level of robustness, cybersecurity and ac..."
1,2,0.725012,SRC-001,AI Act,7,250,which offers practical guidance to help provid...
2,3,0.814218,SRC-001,AI Act,8,250,deepfakes) as well as text. The Guidelines on ...


## Retrieval Quality Inspection

The first retrieval tests show that the vector store is working.

Because the current vector database contains only one source, retrieval quality is limited by source coverage.

Initial observations:

- The risk-based approach query retrieves relevant AI Act overview chunks.
- The high-risk AI systems query retrieves chunks that appear related to high-risk obligations and technical requirements.
- The transparency query retrieves at least one relevant chunk, but the ranking is not yet perfect.

This is expected at this stage.

Retrieval quality should improve after adding the full AI Act text, GDPR sources, and NIST AI RMF content.

In [16]:
def print_retrieval_results(query: str, results_df: pd.DataFrame, max_chars: int = 900) -> None:
    """
    Print retrieval results in a readable format for manual quality inspection.
    """
    print("=" * 100)
    print(f"QUERY: {query}")
    print("=" * 100)
    
    for _, row in results_df.iterrows():
        print()
        print(f"Rank: {row['rank']}")
        print(f"Distance: {row['distance']:.4f}")
        print(f"Source: {row['source_id']} — {row['source_name']}")
        print(f"Chunk index: {row['chunk_index']}")
        print(f"Word count: {row['word_count']}")
        print("-" * 100)
        print(row["retrieved_text"][:max_chars])
        print("-" * 100)


print("Retrieval inspection helper function is ready.")

Retrieval inspection helper function is ready.


In [17]:
print_retrieval_results(query_1, results_query_1)

QUERY: What is the risk-based approach of the EU AI Act?

Rank: 1
Distance: 0.4042
Source: SRC-001 — AI Act
Chunk index: 2
Word count: 250
----------------------------------------------------------------------------------------------------
of the AI Act ahead of time. In parallel, the AI Act Service Desk is also providing information and support for a smooth and effective implementation of the AI Act across the EU. Why do we need rules on AI? The AI Act ensures that Europeans can trust what AI has to offer. While most AI systems pose limited to no risk and can contribute to solving many societal challenges, certain AI systems create risks that we must address to avoid undesirable outcomes. For example, it is often not possible to find out why an AI system has made a decision or prediction and taken a particular action. So, it may become difficult to assess whether someone has been unfairly disadvantaged, such as in a hiring decision or in an application for a public benefit scheme. Alt

In [18]:
print_retrieval_results(query_2, results_query_2)

QUERY: What obligations apply to high-risk AI systems?

Rank: 1
Distance: 0.5887
Source: SRC-001 — AI Act
Chunk index: 2
Word count: 250
----------------------------------------------------------------------------------------------------
of the AI Act ahead of time. In parallel, the AI Act Service Desk is also providing information and support for a smooth and effective implementation of the AI Act across the EU. Why do we need rules on AI? The AI Act ensures that Europeans can trust what AI has to offer. While most AI systems pose limited to no risk and can contribute to solving many societal challenges, certain AI systems create risks that we must address to avoid undesirable outcomes. For example, it is often not possible to find out why an AI system has made a decision or prediction and taken a particular action. So, it may become difficult to assess whether someone has been unfairly disadvantaged, such as in a hiring decision or in an application for a public benefit scheme. Altho

In [19]:
print_retrieval_results(query_3, results_query_3)

QUERY: Does the AI Act require transparency for AI-generated content?

Rank: 1
Distance: 0.6132
Source: SRC-001 — AI Act
Chunk index: 5
Word count: 250
----------------------------------------------------------------------------------------------------
high level of robustness, cybersecurity and accuracy Transparency risk This refers to the risks associated with a need for transparency around the use of AI. The AI Act introduces specific disclosure obligations to ensure that humans are informed when necessary to preserve trust. For instance, when using AI systems such as chatbots, humans should be made aware that they are interacting with a machine so they can take an informed decision. Moreover, providers of generative AI have to ensure that AI-generated content is identifiable. On top of that, certain AI-generated content should be clearly and visibly labelled, namely deep fakes and text published with the purpose to inform the public on matters of public interest. The transparency r

In [20]:
retrieval_test_summary = [
    {
        "test_id": "RT-001",
        "query": query_1,
        "top_chunk_index": int(results_query_1.loc[0, "chunk_index"]),
        "top_distance": float(results_query_1.loc[0, "distance"]),
        "manual_relevance": "good",
        "notes": "Retrieved relevant AI Act overview/risk-based approach content.",
    },
    {
        "test_id": "RT-002",
        "query": query_2,
        "top_chunk_index": int(results_query_2.loc[0, "chunk_index"]),
        "top_distance": float(results_query_2.loc[0, "distance"]),
        "manual_relevance": "acceptable",
        "notes": "Retrieved high-risk AI related content; more precise results expected after adding full AI Act text.",
    },
    {
        "test_id": "RT-003",
        "query": query_3,
        "top_chunk_index": int(results_query_3.loc[0, "chunk_index"]),
        "top_distance": float(results_query_3.loc[0, "distance"]),
        "manual_relevance": "mixed",
        "notes": "At least one retrieved chunk mentions AI-generated content/deepfakes, but ranking can improve with more sources.",
    },
]

retrieval_test_summary_df = pd.DataFrame(retrieval_test_summary)
retrieval_test_summary_df

,test_id,query,top_chunk_index,top_distance,manual_relevance,notes
0,RT-001,What is the risk-based approach of the EU AI Act?,2,0.404169,good,Retrieved relevant AI Act overview/risk-based ...
1,RT-002,What obligations apply to high-risk AI systems?,2,0.588744,acceptable,Retrieved high-risk AI related content; more p...
2,RT-003,Does the AI Act require transparency for AI-ge...,5,0.613211,mixed,At least one retrieved chunk mentions AI-gener...


In [21]:
retrieval_test_summary_path = PROCESSED_DATA_DIR / "retrieval_test_summary_src_001.csv"

retrieval_test_summary_df.to_csv(
    retrieval_test_summary_path,
    index=False,
    encoding="utf-8"
)

print(f"Saved retrieval test summary to: {retrieval_test_summary_path}")
print(f"File exists: {retrieval_test_summary_path.exists()}")
print(f"Rows saved: {len(retrieval_test_summary_df)}")

Saved retrieval test summary to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\retrieval_test_summary_src_001.csv
File exists: True
Rows saved: 3


## Notebook Summary

This notebook created the first embedding and vector-search layer for the Responsible AI RAG Assistant project.

Completed steps:

1. Loaded processed chunks created in Notebook 01.
2. Validated chunk structure and metadata.
3. Loaded the local embedding model `sentence-transformers/all-MiniLM-L6-v2`.
4. Generated a test embedding for one chunk.
5. Generated embeddings for all processed chunks.
6. Validated the embedding matrix.
7. Created a local persistent Chroma vector database.
8. Added chunk documents, metadata, IDs, and embeddings to Chroma.
9. Tested semantic retrieval with three AI governance questions.
10. Created a small manual retrieval quality summary.
11. Saved the retrieval test summary locally.

## Retrieval Quality Notes

The first retrieval tests show that the vector database works.

Initial quality observations:

- The risk-based approach query retrieved relevant AI Act overview content.
- The high-risk AI systems query retrieved acceptable results, but it should improve with the full AI Act legal text.
- The transparency query retrieved partly relevant results, including content related to transparency and deepfakes, but ranking can improve.

This is expected because the vector store currently contains only one source.

The next stage should connect retrieved chunks to a RAG answer-generation pipeline.

In [22]:
# Final notebook checkpoint

vector_store_exists = VECTOR_STORE_DIR.exists()
collection_count = collection.count()

notebook_02_checkpoint = {
    "processed_chunks_loaded": len(chunks_df),
    "embedding_model": embedding_model_name,
    "embedding_dimension": chunk_embeddings.shape[1],
    "embedding_rows": chunk_embeddings.shape[0],
    "vector_store_directory_exists": vector_store_exists,
    "chroma_collection_name": collection_name,
    "chroma_collection_count": collection_count,
    "retrieval_tests_saved": retrieval_test_summary_path.exists(),
    "retrieval_test_rows": len(retrieval_test_summary_df),
}

print("Notebook 02 final checkpoint completed.")
print(f"Processed chunks loaded: {notebook_02_checkpoint['processed_chunks_loaded']}")
print(f"Embedding model: {notebook_02_checkpoint['embedding_model']}")
print(f"Embedding matrix shape: {chunk_embeddings.shape}")
print(f"Chroma collection name: {notebook_02_checkpoint['chroma_collection_name']}")
print(f"Chroma collection count: {notebook_02_checkpoint['chroma_collection_count']}")
print(f"Retrieval test summary exists: {notebook_02_checkpoint['retrieval_tests_saved']}")
print(f"Retrieval test rows: {notebook_02_checkpoint['retrieval_test_rows']}")

Notebook 02 final checkpoint completed.
Processed chunks loaded: 11
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding matrix shape: (11, 384)
Chroma collection name: responsible_ai_rag_src_001
Chroma collection count: 11
Retrieval test summary exists: True
Retrieval test rows: 3


## Next Notebook

The next notebook will be:

`03_rag_question_answering.ipynb`

The goal of the next notebook is to build the first RAG question-answering pipeline.

The next notebook will:

1. Load the existing Chroma vector store.
2. Retrieve relevant chunks for a user question.
3. Format retrieved chunks as context.
4. Create a grounded answer prompt.
5. Generate a source-aware answer.
6. Add responsible AI guardrails.
7. Include a clear disclaimer that the assistant does not provide legal advice.
8. Test the assistant with AI Act, high-risk AI, and transparency questions.

The next stage connects retrieval to answer generation.